# 04a. Modeling Preparation


## 1. Data Loading
Load the preprocessed police call dataset.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    precision_recall_curve,
)

plt.rcParams["figure.figsize"] = (11, 5)

In [2]:
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
# Specifically load the cleaned dataset, ignoring any intermediate modeling files
data_path = PROCESSED_DIR / "sjpd_calls_2016_2026_processed.parquet"

if not data_path.exists():
    raise FileNotFoundError(
        f"Missing cleaned dataset at {data_path}. Please run 02_data_preprocessing.ipynb first."
    )

df = pd.read_parquet(data_path)
print(f"Loaded cleaned shape from {data_path.name}:", df.shape)
display(df.head())


Loaded cleaned shape from sjpd_calls_2016_2026_processed.parquet: (3077538, 26)


,CDTS,EID,START_DATE,CALL_NUMBER,PRIORITY,REPORT_DATE,OFFENSE_DATE,OFFENSE_TIME,CALLTYPE_CODE,CALL_TYPE,...,year,month,hour,day_of_week,is_weekend,time_bin,is_canceled,PRIORITY_NUM,is_priority_1,UNHOUSED
0,2016-05-14 22:22:22,6235632,5/14/2021 12:00:00 AM,P161350913,3,5/14/2016 12:00:00 AM,5/14/2016 12:00:00 AM,21:23:01,SUSCIR,SUSPICIOUS CIRCUMSTANCES,...,2016,5,22,Saturday,True,2016-05-14 22:15:00,False,3,False,NaN
1,2016-05-14 21:44:06,6235633,5/14/2021 12:00:00 AM,P161350914,4,5/14/2016 12:00:00 AM,5/14/2016 12:00:00 AM,21:23:55,415M,"DISTURBANCE, MUSIC",...,2016,5,21,Saturday,True,2016-05-14 21:30:00,False,4,False,NaN
2,2016-05-14 21:26:18,6235635,5/14/2021 12:00:00 AM,P161350915,3,5/14/2016 12:00:00 AM,5/14/2016 12:00:00 AM,21:25:39,1033A,"ALARM, AUDIBLE",...,2016,5,21,Saturday,True,2016-05-14 21:15:00,False,3,False,NaN
3,2016-05-14 22:53:46,6235636,5/14/2021 12:00:00 AM,P161350916,4,5/14/2016 12:00:00 AM,5/14/2016 12:00:00 AM,21:26:34,415M,"DISTURBANCE, MUSIC",...,2016,5,22,Saturday,True,2016-05-14 22:45:00,False,4,False,NaN
4,2016-05-14 22:46:27,6235634,5/14/2021 12:00:00 AM,P161350917,2,5/14/2016 12:00:00 AM,5/14/2016 12:00:00 AM,21:26:48,1181,"VEHICLE ACCIDENT, MINOR INJURI",...,2016,5,22,Saturday,True,2016-05-14 22:45:00,False,2,False,NaN


## 2. Time-Bin Aggregation
Aggregate calls into 15-minute windows and calculate rolling statistics.

In [3]:
time_bin_size = "15min"
df["time_bin"] = df["CDTS"].dt.floor(time_bin_size)

agg = df.groupby("time_bin", as_index=False).agg(
    total_calls=("CDTS", "size"),
    canceled_calls=("is_canceled", "sum"),
    priority_1_calls=("is_priority_1", "sum"),
    unique_call_types=("CALL_TYPE", "nunique"),
)

agg["cancel_rate"] = np.where(
    agg["total_calls"] > 0, agg["canceled_calls"] / agg["total_calls"], 0.0
)
agg["priority_1_share"] = np.where(
    agg["total_calls"] > 0, agg["priority_1_calls"] / agg["total_calls"], 0.0
)

agg["year"] = agg["time_bin"].dt.year
agg["month"] = agg["time_bin"].dt.month
agg["hour"] = agg["time_bin"].dt.hour
agg["day_of_week"] = agg["time_bin"].dt.day_name()
agg["is_weekend"] = (agg["time_bin"].dt.dayofweek >= 5).astype("int8")

agg = agg.sort_values("time_bin").reset_index(drop=True)
display(agg.head())
print("Aggregated windows:", len(agg))

,time_bin,total_calls,canceled_calls,priority_1_calls,unique_call_types,cancel_rate,priority_1_share,year,month,hour,day_of_week,is_weekend
0,2016-01-01 00:00:00,10,8,2,4,0.800000,0.200000,2016,1,0,Friday,0
1,2016-01-01 00:15:00,14,10,0,6,0.714286,0.000000,2016,1,0,Friday,0
2,2016-01-01 00:30:00,18,8,0,8,0.444444,0.000000,2016,1,0,Friday,0
3,2016-01-01 00:45:00,11,3,3,8,0.272727,0.272727,2016,1,0,Friday,0
4,2016-01-01 01:00:00,5,3,0,4,0.600000,0.000000,2016,1,1,Friday,0


Aggregated windows: 358145


## 3. Feature Engineering
Define the target variable and create temporal lag features.

In [4]:
surge_threshold = agg["total_calls"].quantile(0.75)
agg["is_surge"] = (agg["total_calls"] >= surge_threshold).astype("int8")

for lag in [1, 2, 4, 8, 12, 24]:
    agg[f"lag_calls_{lag}"] = agg["total_calls"].shift(lag)

for lag in [1, 4, 8]:
    agg[f"lag_cancel_rate_{lag}"] = agg["cancel_rate"].shift(lag)

for lag in [1, 4, 8]:
    agg[f"lag_priority1_share_{lag}"] = agg["priority_1_share"].shift(lag)

agg["rolling_calls_mean_4"] = agg["total_calls"].shift(1).rolling(4).mean()
agg["rolling_calls_mean_12"] = agg["total_calls"].shift(1).rolling(12).mean()
agg["rolling_calls_std_12"] = agg["total_calls"].shift(1).rolling(12).std()

agg["rolling_cancel_mean_12"] = agg["cancel_rate"].shift(1).rolling(12).mean()
agg["rolling_priority1_mean_12"] = agg["priority_1_share"].shift(1).rolling(12).mean()

agg_model = agg.dropna().copy()

print(f"Surge threshold (Q3): {surge_threshold:.2f} calls per 15-min window")
print("Modeling rows after lag/rolling features:", len(agg_model))
print("Surge class balance:")
display(agg_model["is_surge"].value_counts(normalize=True).rename("proportion"))

Surge threshold (Q3): 11.00 calls per 15-min window
Modeling rows after lag/rolling features: 358121
Surge class balance:


is_surge
0    0.686285
1    0.313715
Name: proportion, dtype: float64

## 4. Data Export
Save the modeling-ready dataset for training.

In [5]:
import os
import joblib
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)
agg_model.to_parquet('data/processed/modeling_ready.parquet')
joblib.dump({'surge_threshold': surge_threshold}, 'models/prep_metadata.joblib')
print('Saved modeling_ready.parquet and prep_metadata.joblib')


Saved modeling_ready.parquet and prep_metadata.joblib
